# Lander Videos

Elise flies by feel.

This notebook records and displays videos for the preserved 253-score 10D SolarSystemLander checkpoint.

## Set Up

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: video-setup # requires: colab-setup

import json
from pathlib import Path

import torch

from hpo.evaluation.video import record_checkpoint_videos
from hpo.solar_system_lander.environment import EnvFactory, World

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise_accel"
CHECKPOINT_DIR = COLAB.drive_study_dir / "best_checkpoints" / STUDY_SERIES_NAME
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.pt"
CHECKPOINT_METADATA_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.json"
VIDEO_DIR = COLAB.drive_study_dir / "videos" / STUDY_SERIES_NAME

ENV_FACTORY = EnvFactory(OBSERVATION_MODE)

metadata = json.loads(CHECKPOINT_METADATA_PATH.read_text())
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"score: {metadata['score']:.1f}")
print(f"trial: {metadata['trial_number']}")
metadata["world_scores"]

## Record Videos

In [ ]:
# cell: record-videos # requires: video-setup

WORLDS = [
    World.MERCURY,
    World.VENUS,
    World.EARTH,
    World.MOON,
    World.MARS,
]
SEEDS = [10_000]

video_paths = record_checkpoint_videos(
    checkpoint_path=CHECKPOINT_PATH,
    environment_factory=ENV_FACTORY,
    worlds=WORLDS,
    seeds=SEEDS,
    output_dir=VIDEO_DIR,
    device=device,
)

video_paths

## Watch Videos

In [ ]:
# cell: watch-videos # requires: record-videos

from IPython.display import Markdown, Video, display

for path in video_paths:
    display(Markdown(f"### {Path(path).name}"))
    display(Video(str(path), embed=True, width=720))